# Multi-Hop Retrieval — FakeEncoder v2 Evaluation

**Before running:**
1. Settings → Accelerator → **T4 GPU**
2. Settings → Internet → **On**
3. Google Drive `musique_data/` folder must contain:
   - `musique_ans_v1.0_dev.jsonl`
   - `musique_ans_v1.0_train.jsonl`
   - `fakencoder_best.pt`  ← trained in train_fakencoder_kaggle.ipynb
   - `model2_best.pt`  ← (optional) trained in train_model2_kaggle.ipynb

**What this notebook runs:**
1. **Collapse diagnostic** — checks if FakeEncoder complement ≈ encode(B) (v1 failure mode)
2. **300-query eval** — MDR vs Graph+cosine vs FULL (complement + Model 2)
3. **Gold-edges oracle** — forces correct hop edges into graph; isolates whether complement SCORING works
4. **Full 2417-query eval** (optional, set MAX_EXAMPLES=None)

**Decision gates:**
- `collapse_sim > 0.95` → FakeEncoder collapsed (same as v1). G+D is necessary.
- `FULL > Graph+cos + 0.01` → complement scoring helps. Proceed with G+D.
- `FULL_gold >> FULL_nogold` → graph coverage is the bottleneck, not complement quality.
- `FULL_gold ≈ Graph+cos_gold` → complement concept broken even with perfect edges.

In [ ]:
# ── [EDIT IF NEEDED] ───────────────────────────────────────────────────────────
REPO_URL        = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME       = "multihop-retrieval"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"   # musique_data folder ID
WORK_DIR        = f"/kaggle/working/{REPO_NAME}/retrieval_v2"
MAX_EXAMPLES    = 300    # change to None for all 2417 dev queries
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
# 1. Verify GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings → Accelerator → T4 GPU")

cc = torch.cuda.get_device_properties(0).major
if cc < 7:
    raise RuntimeError(
        f"GPU is P100 (sm_60) — not supported.\n"
        "FIX: Settings → Accelerator → change to T4 GPU"
    )

print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("CUDA OK")

In [ ]:
# 2. Clone / pull repo + install dependencies
import os

repo_root = f"/kaggle/working/{REPO_NAME}"
if not os.path.isdir(repo_root):
    os.system(f"git clone {REPO_URL} {repo_root}")
else:
    os.system(f"cd {repo_root} && git pull")

os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())

os.system("pip install -q 'transformers>=4.35.0' 'rank_bm25>=0.2.2' 'sentence-transformers>=2.2.0' gdown faiss-cpu")
print("Dependencies ready")

In [ ]:
# 3. Download everything from Google Drive (data + checkpoint)
import os, gdown

download_dir = "/kaggle/working/drive_data"

if not os.path.isdir(download_dir):
    print("Downloading from Google Drive...")
    gdown.download_folder(
        id=DRIVE_FOLDER_ID,
        output=download_dir,
        quiet=False,
        use_cookies=False,
    )
else:
    print("Drive data already downloaded")

print("\nDownloaded files:")
for f in sorted(os.listdir(download_dir)):
    size = os.path.getsize(f"{download_dir}/{f}") / 1e6
    print(f"  {f:45s}  {size:7.1f} MB")

In [ ]:
# 4. Set up file paths (symlinks + checkpoint copy)
import os, shutil

drive = "/kaggle/working/drive_data"

os.makedirs(f"{WORK_DIR}/data/musique", exist_ok=True)
os.makedirs(f"{WORK_DIR}/models",       exist_ok=True)
os.makedirs(f"{WORK_DIR}/cache",        exist_ok=True)
os.makedirs(f"{WORK_DIR}/results",      exist_ok=True)

# MuSiQue JSONL -> symlinks
for fname in ["musique_ans_v1.0_train.jsonl", "musique_ans_v1.0_dev.jsonl"]:
    src = f"{drive}/{fname}"
    dst = f"{WORK_DIR}/data/musique/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"  {fname}: OK")

# FakeEncoder checkpoint (Model 1) — required
ckpt = "fakencoder_best.pt"
src  = f"{drive}/{ckpt}"
dst  = f"{WORK_DIR}/models/{ckpt}"
if os.path.exists(src):
    if not os.path.exists(dst):
        print(f"  Copying {ckpt} ({os.path.getsize(src)/1e6:.0f} MB)...", flush=True)
        shutil.copy(src, dst)
    print(f"  {ckpt}: OK ({os.path.getsize(dst)/1e6:.0f} MB)")
else:
    print(f"  WARNING: {ckpt} not found in Drive — will run MDR-only")

# QueryEncoder checkpoint (Model 2) — optional, enables better query alignment
ckpt2 = "model2_best.pt"
src2  = f"{drive}/{ckpt2}"
dst2  = f"{WORK_DIR}/models/{ckpt2}"
if os.path.exists(src2):
    if not os.path.exists(dst2):
        print(f"  Copying {ckpt2} ({os.path.getsize(src2)/1e6:.0f} MB)...", flush=True)
        shutil.copy(src2, dst2)
    print(f"  {ckpt2}: OK ({os.path.getsize(dst2)/1e6:.0f} MB)")
else:
    print(f"  {ckpt2}: not in Drive — FULL system will use Model 1 for query encoding")

print("\nAll paths ready")

In [ ]:
# 5b. Collapse diagnostic — runs BEFORE full eval (~10 min)
# Checks two things:
#   collapse_sim : cos_sim(complement(A,B), encode_query(B))
#                  > 0.95 = collapsed (same as v1)  |  < 0.85 = differentiated
#   high_g_frac  : fraction of B tokens unreachable from A (g_i > 0.70)
#                  should be > 0.60 for MuSiQue (shortcut-free design)
import os, sys, torch, torch.nn.functional as F
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
sys.path.insert(0, f"{WORK_DIR}/../retrieval")

from transformers import BertTokenizerFast
from fakencoder_train import FakeEncoderModel, MAX_A_LEN, MAX_B_LEN
from data_loader import load_musique, build_chain_quadruples

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ckpt_path = f"{WORK_DIR}/models/fakencoder_best.pt"
if not os.path.exists(ckpt_path):
    print("fakencoder_best.pt not found -- skipping collapse diagnostic")
else:
    model = FakeEncoderModel().to(DEVICE)
    state = torch.load(ckpt_path, map_location=DEVICE)
    incompatible = model.load_state_dict(state, strict=False)
    if incompatible.unexpected_keys:
        print(f"Ignored checkpoint keys: {incompatible.unexpected_keys}")
    model.eval()
    tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

    diag_corpus, diag_queries = load_musique(split="validation", max_examples=150, cache=False)
    diag_quads = build_chain_quadruples(diag_corpus, diag_queries)[:50]
    id_to_text = {c["chunk_id"]: c["text"] for c in diag_corpus}

    collapse_sims, high_g_fracs, bridge_fracs = [], [], []

    with torch.no_grad():
        for quad in diag_quads:
            a_text = id_to_text.get(quad["chunk_a_id"], "")
            b_text = id_to_text.get(quad["chunk_b_pos_id"], "")
            if not a_text or not b_text:
                continue

            enc_a = tokenizer(a_text, max_length=MAX_A_LEN, truncation=True,
                              padding="max_length", return_tensors="pt")
            enc_b = tokenizer(b_text, max_length=MAX_B_LEN, truncation=True,
                              padding="max_length", return_tensors="pt")

            # collapse check: complement vs encode_query(B)
            comp  = model.extract_complement(
                enc_a["input_ids"].to(DEVICE),
                enc_a["attention_mask"].to(DEVICE),
                enc_b["input_ids"].to(DEVICE),
            )
            b_vec = model.encode_query(
                enc_b["input_ids"].to(DEVICE),
                enc_b["attention_mask"].to(DEVICE),
            )
            collapse_sims.append(F.cosine_similarity(comp, b_vec).item())

            # complement gate check (trained BERT cross-attention)
            A_sa = model.encoder1(
                input_ids=enc_a["input_ids"].to(DEVICE),
                attention_mask=enc_a["attention_mask"].to(DEVICE),
            ).last_hidden_state[0]
            B_sa = model.encoder1(
                input_ids=enc_b["input_ids"].to(DEVICE),
                attention_mask=enc_b["attention_mask"].to(DEVICE),
            ).last_hidden_state[0]
            a_real = enc_a["attention_mask"][0].bool()
            b_real = enc_b["attention_mask"][0].bool()

            d = B_sa.size(-1)
            scores = torch.matmul(B_sa[b_real], A_sa[a_real].T) / (d ** 0.5)
            attn_w = torch.softmax(scores, dim=-1)
            g_i = 1.0 - attn_w.max(dim=-1).values
            high_g_fracs.append((g_i > 0.70).float().mean().item())
            bridge_fracs.append((g_i < 0.30).float().mean().item())

    mean_collapse = sum(collapse_sims) / max(len(collapse_sims), 1)
    mean_high_g   = sum(high_g_fracs)  / max(len(high_g_fracs), 1)
    mean_bridge   = sum(bridge_fracs)  / max(len(bridge_fracs), 1)

    print("=" * 55)
    print("  COLLAPSE DIAGNOSTIC")
    print("=" * 55)
    print(f"  Pairs evaluated           : {len(collapse_sims)}")
    print(f"  collapse_sim  (pass <0.85) : {mean_collapse:.4f}  "
          f"{'OK' if mean_collapse < 0.85 else 'COLLAPSED'}")
    print(f"  high_g_frac   (pass >0.60) : {mean_high_g:.4f}  "
          f"{'OK' if mean_high_g > 0.60 else 'LOW'}")
    print(f"  bridge_frac   (pass <0.10) : {mean_bridge:.4f}  "
          f"{'OK' if mean_bridge < 0.10 else 'HIGH'}")
    print()
    if mean_collapse > 0.95:
        print("  STATUS: COLLAPSED -- complement ~= encode(B)")
        print("  Same failure as v1. G+D is necessary.")
    elif mean_collapse > 0.85:
        print("  STATUS: PARTIAL COLLAPSE -- some differentiation but weak")
        print("  FULL may marginally beat Graph+cos. G+D will improve further.")
    else:
        print("  STATUS: DIFFERENTIATED -- complement moved away from encode(B)")
        print("  Expect FULL > Graph+cos. Complement is working.")
    print("=" * 55)

In [ ]:
# 6. Standard eval (no gold edges)
import sys, os, importlib
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
sys.path.insert(0, f"{WORK_DIR}/../retrieval")

# Force-reload run_eval so any git-pulled changes take effect
# (Python caches imported modules — this ensures we use the latest version)
if "run_eval" in sys.modules:
    importlib.reload(sys.modules["run_eval"])

from run_eval import main
results_standard = main(max_examples=MAX_EXAMPLES)

In [ ]:
# 8. Final comparison table
def r10(d): return d.get("recall@10", 0)
def r5(d):  return d.get("recall@5",  0)

mdr_key  = "MDR (dense, iterative)"
cos_key  = "Graph + direct cosine (FE)"
full_key = "FULL: FE graph + complement + FAISS"

W = 60
print("=" * W)
print("  RESULTS SUMMARY")
print("=" * W)
print()
print("  Standard eval (no gold edges):")
print(f"    MDR           R@5={r5(results_standard.get(mdr_key, {})):.3f}  R@10={r10(results_standard.get(mdr_key, {})):.3f}")
print(f"    Graph+cosine  R@5={r5(results_standard.get(cos_key, {})):.3f}  R@10={r10(results_standard.get(cos_key, {})):.3f}")
print(f"    FULL          R@5={r5(results_standard.get(full_key, {})):.3f}  R@10={r10(results_standard.get(full_key, {})):.3f}")
print()
print("  Gold-edges oracle (correct edges forced into graph):")
print(f"    Graph+cosine  R@5={r5(results_gold.get(cos_key, {})):.3f}  R@10={r10(results_gold.get(cos_key, {})):.3f}")
print(f"    FULL          R@5={r5(results_gold.get(full_key, {})):.3f}  R@10={r10(results_gold.get(full_key, {})):.3f}")
print()

full_std  = r10(results_standard.get(full_key, {}))
full_gold = r10(results_gold.get(full_key, {}))
cos_std   = r10(results_standard.get(cos_key, {}))
cos_gold  = r10(results_gold.get(cos_key, {}))
mdr_r10   = r10(results_standard.get(mdr_key, {}))

oracle_gain       = full_gold - full_std
complement_vs_cos = full_std  - cos_std
gold_comp_vs_cos  = full_gold - cos_gold

print("=" * W)
print("  DECISION ANALYSIS")
print("=" * W)
print(f"  FULL vs Graph+cosine (standard)  : {complement_vs_cos:+.4f}"
      f"  ({'complement helps' if complement_vs_cos > 0.01 else 'marginal' if complement_vs_cos > 0 else 'hurts'})")
print(f"  FULL gold vs standard (oracle)   : {oracle_gain:+.4f}"
      f"  ({'graph coverage was bottleneck' if oracle_gain > 0.03 else 'coverage fine' if oracle_gain > 0.01 else 'no gap'})")
print(f"  FULL gold vs Graph+cos gold      : {gold_comp_vs_cos:+.4f}"
      f"  ({'complement adds value' if gold_comp_vs_cos > 0.01 else 'complement marginal'})")
print(f"  FULL vs MDR                      : {full_std - mdr_r10:+.4f}"
      f"  ({'beats MDR' if full_std > mdr_r10 else 'below MDR'})")
print()
print("=" * W)
print("  NEXT STEP")
print("=" * W)
if complement_vs_cos > 0.01 and gold_comp_vs_cos > 0.01:
    print("  Complement WORKS. Proceed with G+D architecture.")
    print("  G+D will produce better edges than FakeEncoder v2.")
elif full_gold > full_std + 0.03:
    print("  Graph coverage is main bottleneck.")
    print("  Complement scores correctly when edges exist.")
    print("  Proceed with G+D -- also improve graph construction.")
elif full_gold <= cos_gold + 0.005:
    print("  Complement scoring adds nothing even with perfect edges.")
    print("  Reconsider pipeline before implementing G+D.")
else:
    print("  Marginal complement signal. G+D should improve it significantly.")
    print("  Proceed with G+D implementation.")
print("=" * W)

In [ ]:
# 7. Gold-edges oracle eval (~10 min, reuses cached embeddings)
results_gold = main(max_examples=MAX_EXAMPLES, gold_edges=True)

In [ ]:
# 5. Run evaluation
import sys, os
os.chdir(WORK_DIR)

# Add retrieval_v2/ and retrieval/ to path so imports resolve correctly
sys.path.insert(0, WORK_DIR)
sys.path.insert(0, f"{WORK_DIR}/../retrieval")

from run_eval import main
results = main(max_examples=MAX_EXAMPLES)

In [ ]:
# 6. Summary table
print("\n=== FINAL RESULTS ===")
for name, metrics in results.items():
    r10 = metrics.get('recall@10', 0)
    r5  = metrics.get('recall@5',  0)
    r2  = metrics.get('recall@2',  0)
    print(f"  {name:45s}  R@2={r2:.3f}  R@5={r5:.3f}  R@10={r10:.3f}")